# Сравнение бэкендов CLIP-ReID на MSMT17

Бэкенд | Точность | Железо
---|---|---
1. PyTorch FP32    | FP32         | Tesla V100 PCIe 32GB
2. ONNX CPU FP32   | FP32         | 4 cores, 64 GB RAM
3. ONNX CUDA FP32  | FP32         | Tesla V100 PCIe 32GB
4. ONNX CUDA FP16  | FP16         | Tesla V100 PCIe 32GB
5. ONNX CPU INT8   | INT8 (dyn.)  | 4 cores, 64 GB RAM
6. TensorRT FP16   | FP16         | Tesla V100 PCIe 32GB
7. TensorRT INT8   | INT8 (calib) | Tesla V100 PCIe 32GB

Метрики: **mAP · Recall@K · Precision@K · F2@K · Time**

Стратегия нагрузки GPU:
- Batch size подбирается автоматически под 18 GB свободной VRAM
- DataLoader: `num_workers=4`, `pin_memory=True`, `prefetch_factor=2`
- Матрица расстояний [11 659 × 82 161] вычисляется на GPU целиком (~3.6 GB)
- Для каждого GPU-бэкенда выполняется прогрев 3 батчами перед замером

In [ ]:
import os, sys, time, gc
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

REPO_DIR = os.path.dirname(os.path.abspath("__file__"))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from utils.metrics import eval_func

assert torch.cuda.is_available(), "CUDA недоступна"
DEVICE = torch.device("cuda:0")
gpu_props = torch.cuda.get_device_properties(0)
total_vram = gpu_props.total_memory / 1024**3
print(f"GPU       : {gpu_props.name}")
print(f"VRAM total: {total_vram:.1f} GB")
print(f"torch     : {torch.__version__}")

## Настройка

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    НАСТРОЙТЕ ПОД СЕРВЕР                     ║
# ╚══════════════════════════════════════════════════════════════╝

DATASET_ROOT  = "../../MSMT17_V1"
CONFIG_PATH   = "../clip-reid/configs/person/vit_clipreid.yml"
PTH_PATH      = "../clip-reid/MSMT17_clipreid_ViT-B-16_60.pth"

# ONNX модели
ONNX_PATH      = "clipreid_msmt17.onnx"          # FP32 (базовый)
ONNX_FP16_PATH = "clipreid_msmt17_fp16.onnx"     # FP16 (конвертируется ниже)
ONNX_INT8_PATH = "clipreid_msmt17_int8.onnx"     # INT8 (конвертируется ниже)

# TensorRT движки
TRT_PATH      = "clipreid_msmt17.trt"             # TRT FP16
TRT_INT8_PATH = "clipreid_msmt17_int8.trt"        # TRT INT8 (из convert_to_onnx_trt.ipynb)

NUM_WORKERS   = 4       # CPU ядра для DataLoader
VRAM_BUDGET   = 18.0    # GB свободной VRAM для батча
NUM_CLASSES   = 1041
CAMERA_NUM    = 15

# Автоматический расчёт max batch size по бюджету VRAM
# [B, 3, 256, 128] float32 = B * 393216 байт
# Берём 30% бюджета на входные данные, остальное — на активации
BYTES_PER_IMG  = 3 * 256 * 128 * 4
MAX_BATCH_SIZE = int((VRAM_BUDGET * 0.30 * 1024**3) / BYTES_PER_IMG)
MAX_BATCH_SIZE = min(MAX_BATCH_SIZE, 512)   # разумный предел
print(f"Авто batch size : {MAX_BATCH_SIZE}")

TEST_DIR          = os.path.join(DATASET_ROOT, "test")
LIST_QUERY_PATH   = os.path.join(DATASET_ROOT, "list_query.txt")
LIST_GALLERY_PATH = os.path.join(DATASET_ROOT, "list_gallery.txt")

## Подготовка FP16 и INT8 моделей

| Формат | Инструмент | Особенность |
|---|---|---|
| ONNX FP16 | `onnxconverter-common` | Веса и активации в float16; вход/выход остаётся float32 (`keep_io_types=True`) |
| ONNX INT8 | `onnxruntime.quantization` | Динамическая квантизация: веса INT8, активации на лету; оптимально для CPU с AVX-512/VNNI |
| TRT INT8 | TensorRT calibration | Требует калибровочный датасет; собирается в `convert_to_onnx_trt.ipynb` |

> Ячейка ниже выполняется **один раз** — файлы сохраняются на диск и при повторном запуске пропускаются.

In [ ]:
import onnx
from onnxconverter_common import float16 as onnx_float16
from onnxruntime.quantization import quantize_dynamic, QuantType

# ──────────────────────────────────────────────────────────────────────────────
# ONNX FP32 → FP16
# pip install onnxconverter-common onnx
# keep_io_types=True: вход/выход остаётся float32 — не нужно менять препроцессинг
# ──────────────────────────────────────────────────────────────────────────────
if not os.path.exists(ONNX_FP16_PATH):
    print(f"Конвертация {ONNX_PATH} → {ONNX_FP16_PATH} ...")
    model_fp32 = onnx.load(ONNX_PATH)
    model_fp16 = onnx_float16.convert_float_to_float16(model_fp32, keep_io_types=True)
    onnx.save(model_fp16, ONNX_FP16_PATH)
    print(f"  Готово: {ONNX_FP16_PATH}  ({os.path.getsize(ONNX_FP16_PATH)/1024**2:.1f} MB)")
else:
    print(f"FP16 уже есть: {ONNX_FP16_PATH}  ({os.path.getsize(ONNX_FP16_PATH)/1024**2:.1f} MB)")

# ──────────────────────────────────────────────────────────────────────────────
# ONNX FP32 → INT8  (динамическая квантизация)
# Веса квантизируются в INT8; активации квантизируются на лету во время инференса.
# Хорошо работает на CPU с AVX-512 (VNNI).
# ──────────────────────────────────────────────────────────────────────────────
if not os.path.exists(ONNX_INT8_PATH):
    print(f"\nКвантизация {ONNX_PATH} → {ONNX_INT8_PATH} ...")
    quantize_dynamic(
        ONNX_PATH,
        ONNX_INT8_PATH,
        weight_type=QuantType.QInt8,
        per_channel=False,   # False — быстрее; True — точнее, но требует reshape
        reduce_range=True,   # безопаснее для VNNI / AVX-512
    )
    print(f"  Готово: {ONNX_INT8_PATH}  ({os.path.getsize(ONNX_INT8_PATH)/1024**2:.1f} MB)")
else:
    print(f"INT8 уже есть: {ONNX_INT8_PATH}  ({os.path.getsize(ONNX_INT8_PATH)/1024**2:.1f} MB)")

# ──────────────────────────────────────────────────────────────────────────────
# TRT INT8
# Строится в convert_to_onnx_trt.ipynb (требует калибровочный датасет).
# ──────────────────────────────────────────────────────────────────────────────
if os.path.exists(TRT_INT8_PATH):
    print(f"\nTRT INT8 движок найден: {TRT_INT8_PATH}  ({os.path.getsize(TRT_INT8_PATH)/1024**2:.1f} MB)")
else:
    print(f"\n⚠  TRT INT8 движок не найден: {TRT_INT8_PATH}")
    print("   Запустите ячейку 'TRT INT8 Build' в convert_to_onnx_trt.ipynb")

## Датасет и парсинг

In [ ]:
TRANSFORM = T.Compose([
    T.Resize((256, 128), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(
        mean=[0.48145466, 0.4578275,  0.40821073],
        std= [0.26862954, 0.26130258, 0.27577711],
    )
])


def parse_list(list_path: str, img_dir: str) -> list:
    records = []
    with open(list_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rel_path, pid_str = line.split(" ")
            pid   = int(pid_str)
            fname = os.path.basename(rel_path)
            camid = int(fname.split("_")[2])
            records.append((os.path.join(img_dir, rel_path), pid, camid))
    return records


class ReIDDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, pid, camid = self.records[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), pid, camid


query_records   = parse_list(LIST_QUERY_PATH,   TEST_DIR)
gallery_records = parse_list(LIST_GALLERY_PATH, TEST_DIR)

q_pids   = np.array([r[1] for r in query_records])
q_camids = np.array([r[2] for r in query_records])
g_pids   = np.array([r[1] for r in gallery_records])
g_camids = np.array([r[2] for r in gallery_records])

print(f"Query   : {len(query_records):6d} imgs | {len(np.unique(q_pids))} IDs")
print(f"Gallery : {len(gallery_records):6d} imgs | {len(np.unique(g_pids))} IDs")


def make_loader(records, batch_size: int) -> DataLoader:
    """DataLoader с pin_memory и prefetch для максимальной пропускной способности."""
    return DataLoader(
        ReIDDataset(records, TRANSFORM),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=2,
        persistent_workers=True,
        drop_last=False,
    )

## Метрики

In [ ]:
# ────────────────────────────────────────────────────────────
# Базовый класс
# ────────────────────────────────────────────────────────────
class ReIDBackend:
    name: str
    batch_size: int

    def warmup(self, n: int = 3):
        """Прогрев: n батчей, чтобы JIT/TRT кэш был готов."""
        dummy = np.random.randn(self.batch_size, 3, 256, 128).astype(np.float32)
        for _ in range(n):
            self._infer_batch(dummy)
        torch.cuda.synchronize()

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def extract(self, records: list) -> tuple:
        """
        Извлекает эмбеддинги для всего списка записей.

        Returns:
            embs   : [N, EMB_DIM] float32
            total_s: секунд чистого инференса (без I/O)
        """
        loader = make_loader(records, self.batch_size)
        all_embs = []
        infer_time = 0.0

        for imgs, _, _ in tqdm(loader, desc=f"  {self.name}", unit="batch"):
            imgs_np = imgs.numpy()  # pin_memory → numpy без копирования

            torch.cuda.synchronize()
            t0 = time.perf_counter()
            embs = self._infer_batch(imgs_np)
            torch.cuda.synchronize()
            infer_time += time.perf_counter() - t0

            all_embs.append(embs)

        embs = np.concatenate(all_embs, axis=0).astype(np.float32)
        # L2-нормализация
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        embs  = embs / np.clip(norms, 1e-8, None)
        return embs, infer_time

    def close(self):
        pass


# ────────────────────────────────────────────────────────────
# 1. PyTorch FP32
# ────────────────────────────────────────────────────────────
class PyTorchBackend(ReIDBackend):
    name = "PyTorch FP32"

    def __init__(self, config_path, weights_path, batch_size, num_class, camera_num):
        from config import cfg
        from model.make_model_clipreid import make_model

        cfg.defrost()
        cfg.merge_from_file(config_path)
        cfg.freeze()

        self.model = make_model(cfg, num_class=num_class, camera_num=camera_num, view_num=0)
        self.model.load_param(weights_path)
        self.model.eval().cuda()
        self.batch_size = batch_size

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        x = torch.from_numpy(imgs_np).cuda()
        with torch.no_grad():
            feat = self.model(x, cam_label=None, view_label=None)
            if isinstance(feat, (tuple, list)):
                feat = feat[-1]
        return feat.float().cpu().numpy()

    def close(self):
        del self.model
        torch.cuda.empty_cache()
        gc.collect()


# ────────────────────────────────────────────────────────────
# 2. ONNX Runtime — CPU FP32
# ────────────────────────────────────────────────────────────
class ONNXCPUBackend(ReIDBackend):
    name = "ONNX CPU FP32"

    def __init__(self, onnx_path, batch_size):
        import onnxruntime as ort
        opts = ort.SessionOptions()
        opts.intra_op_num_threads  = 4
        opts.inter_op_num_threads  = 1
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session    = ort.InferenceSession(onnx_path, opts,
                                               providers=["CPUExecutionProvider"])
        self.batch_size = batch_size

    def warmup(self, n=3):
        dummy = np.random.randn(self.batch_size, 3, 256, 128).astype(np.float32)
        for _ in range(n):
            self._infer_batch(dummy)

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        return self.session.run(["embedding"], {"image": imgs_np})[0]

    def close(self):
        del self.session
        gc.collect()


# ────────────────────────────────────────────────────────────
# 3. ONNX Runtime — CUDA FP32
# ────────────────────────────────────────────────────────────
class ONNXCUDABackend(ReIDBackend):
    name = "ONNX CUDA FP32"

    def __init__(self, onnx_path, batch_size):
        import onnxruntime as ort
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session = ort.InferenceSession(
            onnx_path, opts,
            providers=[
                ("CUDAExecutionProvider", {
                    "device_id": 0,
                    "arena_extend_strategy": "kNextPowerOfTwo",
                    "cudnn_conv_algo_search": "EXHAUSTIVE",
                    "do_copy_in_default_stream": True,
                }),
                "CPUExecutionProvider",
            ]
        )
        self.batch_size = batch_size
        active = self.session.get_providers()[0]
        print(f"  ORT провайдер: {active}")

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        return self.session.run(["embedding"], {"image": imgs_np})[0]

    def close(self):
        del self.session
        torch.cuda.empty_cache()
        gc.collect()


# ────────────────────────────────────────────────────────────
# 4. ONNX Runtime — CUDA FP16
# Загружает FP16-модель (сконвертированную выше).
# keep_io_types=True при конвертации → вход/выход float32,
# внутренние операции в float16.
# ────────────────────────────────────────────────────────────
class ONNXCUDAFp16Backend(ReIDBackend):
    name = "ONNX CUDA FP16"

    def __init__(self, onnx_fp16_path, batch_size):
        import onnxruntime as ort
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session = ort.InferenceSession(
            onnx_fp16_path, opts,
            providers=[
                ("CUDAExecutionProvider", {
                    "device_id": 0,
                    "arena_extend_strategy": "kNextPowerOfTwo",
                    "cudnn_conv_algo_search": "EXHAUSTIVE",
                    "do_copy_in_default_stream": True,
                }),
                "CPUExecutionProvider",
            ]
        )
        self.batch_size = batch_size
        inp = self.session.get_inputs()[0]
        # Если keep_io_types=True → вход float32; иначе float16
        self._input_dtype = np.float16 if "float16" in inp.type else np.float32
        active = self.session.get_providers()[0]
        print(f"  ORT провайдер: {active}  | вход: {inp.type}")

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        inp = imgs_np.astype(self._input_dtype)
        out = self.session.run(["embedding"], {"image": inp})[0]
        return out.astype(np.float32)

    def close(self):
        del self.session
        torch.cuda.empty_cache()
        gc.collect()


# ────────────────────────────────────────────────────────────
# 5. ONNX Runtime — CPU INT8  (динамическая квантизация)
# Веса в INT8; активации квантизируются at runtime.
# Максимальный эффект на CPU с AVX-512 VNNI.
# ────────────────────────────────────────────────────────────
class ONNXCPUInt8Backend(ReIDBackend):
    name = "ONNX CPU INT8"

    def __init__(self, onnx_int8_path, batch_size):
        import onnxruntime as ort
        opts = ort.SessionOptions()
        opts.intra_op_num_threads  = 4
        opts.inter_op_num_threads  = 1
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session    = ort.InferenceSession(onnx_int8_path, opts,
                                               providers=["CPUExecutionProvider"])
        self.batch_size = batch_size

    def warmup(self, n=3):
        dummy = np.random.randn(self.batch_size, 3, 256, 128).astype(np.float32)
        for _ in range(n):
            self._infer_batch(dummy)

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        return self.session.run(["embedding"], {"image": imgs_np})[0]

    def close(self):
        del self.session
        gc.collect()


# ────────────────────────────────────────────────────────────
# 6. TensorRT FP16
# ────────────────────────────────────────────────────────────
class TRTBackend(ReIDBackend):
    name = "TensorRT FP16"

    def __init__(self, trt_path, batch_size):
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit

        self._trt  = trt
        self._cuda = cuda
        logger     = trt.Logger(trt.Logger.WARNING)
        runtime    = trt.Runtime(logger)

        with open(trt_path, "rb") as f:
            self.engine  = runtime.deserialize_cuda_engine(f.read())
        self.context = self.engine.create_execution_context()
        self.stream  = cuda.Stream()
        self.batch_size = batch_size
        self._alloc_buffers()

    def _alloc_buffers(self):
        self._inputs  = []
        self._outputs = []
        for i in range(self.engine.num_io_tensors):
            name  = self.engine.get_tensor_name(i)
            dtype = self._trt.nptype(self.engine.get_tensor_dtype(name))
            mode  = self.engine.get_tensor_mode(name)
            max_s = self.engine.get_tensor_profile_shape(name, 0)[2]
            h_mem = self._cuda.pagelocked_empty(int(np.prod(max_s)), dtype)
            d_mem = self._cuda.mem_alloc(h_mem.nbytes)
            entry = {"name": name, "host": h_mem, "device": d_mem,
                     "dtype": dtype, "max_shape": max_s}
            (self._inputs if mode == self._trt.TensorIOMode.INPUT
             else self._outputs).append(entry)

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        b    = imgs_np.shape[0]
        self.context.set_input_shape("image", imgs_np.shape)
        np.copyto(self._inputs[0]["host"][:imgs_np.size], imgs_np.ravel())
        self._cuda.memcpy_htod_async(self._inputs[0]["device"],
                                     self._inputs[0]["host"], self.stream)
        self.context.execute_async_v3(stream_handle=self.stream.handle)
        self._cuda.memcpy_dtoh_async(self._outputs[0]["host"],
                                     self._outputs[0]["device"], self.stream)
        self.stream.synchronize()
        emb_dim = self._outputs[0]["max_shape"][-1]
        return self._outputs[0]["host"][:b * emb_dim].reshape(b, emb_dim).copy()

    def close(self):
        del self.context, self.engine
        for buf in self._inputs + self._outputs:
            buf["device"].free()
        torch.cuda.empty_cache()
        gc.collect()


# ────────────────────────────────────────────────────────────
# 7. TensorRT INT8
# Идентичен TRTBackend — отличается только движком.
# Движок строится в convert_to_onnx_trt.ipynb
# (требует калибровочный датасет из train MSMT17).
# ────────────────────────────────────────────────────────────
class TRTInt8Backend(TRTBackend):
    name = "TensorRT INT8"


print("Классы бэкендов определены ✓  (7 шт.)")

## Бэкенды

In [ ]:
# ────────────────────────────────────────────────────────────
# Базовый класс
# ────────────────────────────────────────────────────────────
class ReIDBackend:
    name: str
    batch_size: int

    def warmup(self, n: int = 3):
        """Прогрев: n батчей, чтобы JIT/TRT кэш был готов."""
        dummy = np.random.randn(self.batch_size, 3, 256, 128).astype(np.float32)
        for _ in range(n):
            self._infer_batch(dummy)
        torch.cuda.synchronize()

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def extract(self, records: list) -> tuple:
        """
        Извлекает эмбеддинги для всего списка записей.

        Returns:
            embs   : [N, EMB_DIM] float32
            total_s: секунд чистого инференса (без I/O)
        """
        loader = make_loader(records, self.batch_size)
        all_embs = []
        infer_time = 0.0

        for imgs, _, _ in tqdm(loader, desc=f"  {self.name}", unit="batch"):
            imgs_np = imgs.numpy()  # pin_memory → numpy без копирования

            torch.cuda.synchronize()
            t0 = time.perf_counter()
            embs = self._infer_batch(imgs_np)
            torch.cuda.synchronize()
            infer_time += time.perf_counter() - t0

            all_embs.append(embs)

        embs = np.concatenate(all_embs, axis=0).astype(np.float32)
        # L2-нормализация
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        embs  = embs / np.clip(norms, 1e-8, None)
        return embs, infer_time

    def close(self):
        pass


# ────────────────────────────────────────────────────────────
# 1. PyTorch FP32
# ────────────────────────────────────────────────────────────
class PyTorchBackend(ReIDBackend):
    name = "PyTorch FP32"

    def __init__(self, config_path, weights_path, batch_size, num_class, camera_num):
        from config import cfg
        from model.make_model_clipreid import make_model

        cfg.defrost()
        cfg.merge_from_file(config_path)
        cfg.freeze()

        self.model = make_model(cfg, num_class=num_class, camera_num=camera_num, view_num=0)
        self.model.load_param(weights_path)
        self.model.eval().cuda()
        self.batch_size = batch_size

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        x = torch.from_numpy(imgs_np).cuda()
        with torch.no_grad():
            feat = self.model(x, cam_label=None, view_label=None)
            if isinstance(feat, (tuple, list)):
                feat = feat[-1]
        return feat.float().cpu().numpy()

    def close(self):
        del self.model
        torch.cuda.empty_cache()
        gc.collect()


# ────────────────────────────────────────────────────────────
# 2. ONNX Runtime — CPU
# ────────────────────────────────────────────────────────────
class ONNXCPUBackend(ReIDBackend):
    name = "ONNX CPU"

    def __init__(self, onnx_path, batch_size):
        import onnxruntime as ort
        opts = ort.SessionOptions()
        opts.intra_op_num_threads  = 4
        opts.inter_op_num_threads  = 1
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session    = ort.InferenceSession(onnx_path, opts,
                                               providers=["CPUExecutionProvider"])
        self.batch_size = batch_size

    def warmup(self, n=3):
        dummy = np.random.randn(self.batch_size, 3, 256, 128).astype(np.float32)
        for _ in range(n):
            self._infer_batch(dummy)

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        return self.session.run(["embedding"], {"image": imgs_np})[0]

    def close(self):
        del self.session
        gc.collect()


# ────────────────────────────────────────────────────────────
# 3. ONNX Runtime — CUDA
# ────────────────────────────────────────────────────────────
class ONNXCUDABackend(ReIDBackend):
    name = "ONNX CUDA"

    def __init__(self, onnx_path, batch_size):
        import onnxruntime as ort
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session = ort.InferenceSession(
            onnx_path, opts,
            providers=[
                ("CUDAExecutionProvider", {
                    "device_id": 0,
                    "arena_extend_strategy": "kNextPowerOfTwo",
                    "cudnn_conv_algo_search": "EXHAUSTIVE",
                    "do_copy_in_default_stream": True,
                }),
                "CPUExecutionProvider",
            ]
        )
        self.batch_size = batch_size
        active = self.session.get_providers()[0]
        print(f"  ORT провайдер: {active}")

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        return self.session.run(["embedding"], {"image": imgs_np})[0]

    def close(self):
        del self.session
        torch.cuda.empty_cache()
        gc.collect()


# ────────────────────────────────────────────────────────────
# 4. TensorRT FP16
# ────────────────────────────────────────────────────────────
class TRTBackend(ReIDBackend):
    name = "TensorRT FP16"

    def __init__(self, trt_path, batch_size):
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit

        self._trt  = trt
        self._cuda = cuda
        logger     = trt.Logger(trt.Logger.WARNING)
        runtime    = trt.Runtime(logger)

        with open(trt_path, "rb") as f:
            self.engine  = runtime.deserialize_cuda_engine(f.read())
        self.context = self.engine.create_execution_context()
        self.stream  = cuda.Stream()
        self.batch_size = batch_size
        self._alloc_buffers()

    def _alloc_buffers(self):
        self._inputs  = []
        self._outputs = []
        import numpy as np
        for i in range(self.engine.num_io_tensors):
            name  = self.engine.get_tensor_name(i)
            dtype = self._trt.nptype(self.engine.get_tensor_dtype(name))
            mode  = self.engine.get_tensor_mode(name)
            max_s = self.engine.get_tensor_profile_shape(name, 0)[2]
            h_mem = self._cuda.pagelocked_empty(int(np.prod(max_s)), dtype)
            d_mem = self._cuda.mem_alloc(h_mem.nbytes)
            entry = {"name": name, "host": h_mem, "device": d_mem,
                     "dtype": dtype, "max_shape": max_s}
            (self._inputs if mode == self._trt.TensorIOMode.INPUT
             else self._outputs).append(entry)

    def _infer_batch(self, imgs_np: np.ndarray) -> np.ndarray:
        b    = imgs_np.shape[0]
        self.context.set_input_shape("image", imgs_np.shape)
        np.copyto(self._inputs[0]["host"][:imgs_np.size], imgs_np.ravel())
        self._cuda.memcpy_htod_async(self._inputs[0]["device"],
                                     self._inputs[0]["host"], self.stream)
        self.context.execute_async_v3(stream_handle=self.stream.handle)
        self._cuda.memcpy_dtoh_async(self._outputs[0]["host"],
                                     self._outputs[0]["device"], self.stream)
        self.stream.synchronize()
        emb_dim = self._outputs[0]["max_shape"][-1]
        return self._outputs[0]["host"][:b * emb_dim].reshape(b, emb_dim).copy()

    def close(self):
        del self.context, self.engine
        for buf in self._inputs + self._outputs:
            buf["device"].free()
        torch.cuda.empty_cache()
        gc.collect()


print("Классы бэкендов определены ✓")

## Прогон бенчмарка

In [ ]:
def run_single_benchmark(backend: ReIDBackend,
                         distance: str = "euclidean") -> dict:
    """
    Полный прогон для одного бэкенда:
    1. Прогрев
    2. Извлечение эмбеддингов query + gallery (с замером)
    3. Вычисление метрик
    4. Сохранение эмбеддингов для последующего сравнения дистанций
    """
    sep = "─" * 55
    print(f"\n{sep}")
    print(f"  {backend.name}  (batch={backend.batch_size})")
    print(sep)

    print("  Прогрев...")
    backend.warmup(n=3)

    wall_start = time.time()
    q_embs, q_infer_s = backend.extract(query_records)
    g_embs, g_infer_s = backend.extract(gallery_records)
    wall_total    = time.time() - wall_start
    infer_total_s = q_infer_s + g_infer_s
    n_images      = len(query_records) + len(gallery_records)
    ms_per_img    = infer_total_s * 1000 / n_images
    throughput    = n_images / infer_total_s

    print(f"  Инференс: {infer_total_s:.2f} сек | {ms_per_img:.3f} ms/img | {throughput:.0f} img/s")
    print(f"  Wall time: {wall_total:.2f} сек")

    metrics = compute_all_metrics(q_embs, g_embs, q_pids, g_pids,
                                  q_camids, g_camids, distance=distance)

    result = {
        "name":       backend.name,
        "batch_size": backend.batch_size,
        "infer_s":    infer_total_s,
        "ms_per_img": ms_per_img,
        "throughput": throughput,
        "wall_s":     wall_total,
        "mAP":        metrics["mAP"],
        "rank1":      metrics["cmc"][0],
        "rank5":      metrics["cmc"][4],
        "rank10":     metrics["cmc"][9],
        "prf":        metrics["prf"],
        # Сохраняем эмбеддинги для сравнения метрик расстояния
        "q_embs":     q_embs,
        "g_embs":     g_embs,
    }

    print(f"\n  mAP={result['mAP']*100:.2f}%  R1={result['rank1']*100:.2f}%  "
          f"P@1={result['prf'][1]['precision']*100:.2f}%  "
          f"F2@1={result['prf'][1]['f2']*100:.2f}%")
    return result


ALL_RESULTS = {}
print("Готово — запускайте ячейки с каждым бэкендом по очереди")

In [ ]:
# ── Бэкенд 1: PyTorch FP32 ────────────────────────────────────────────────
b1 = PyTorchBackend(
    config_path=CONFIG_PATH,
    weights_path=PTH_PATH,
    batch_size=MAX_BATCH_SIZE,
    num_class=NUM_CLASSES,
    camera_num=CAMERA_NUM,
)
ALL_RESULTS["pytorch"] = run_single_benchmark(b1)
b1.close()

In [ ]:
# ── Бэкенд 2: ONNX CPU ───────────────────────────────────────────────────
# CPU батч меньше — загрузка 4 ядер
CPU_BATCH = 64
b2 = ONNXCPUBackend(ONNX_PATH, batch_size=CPU_BATCH)
ALL_RESULTS["onnx_cpu"] = run_single_benchmark(b2)
b2.close()

In [ ]:
# ── Бэкенд 5: ONNX CUDA FP16 ─────────────────────────────────────────────────
b5 = ONNXCUDAFp16Backend(ONNX_FP16_PATH, batch_size=MAX_BATCH_SIZE)
ALL_RESULTS["onnx_cuda_fp16"] = run_single_benchmark(b5)
b5.close()

In [ ]:
# ── Бэкенд 6: ONNX CPU INT8 ──────────────────────────────────────────────────
# INT8 эффективнее на CPU с VNNI; батч такой же, как у ONNX CPU FP32
b6 = ONNXCPUInt8Backend(ONNX_INT8_PATH, batch_size=CPU_BATCH)
ALL_RESULTS["onnx_cpu_int8"] = run_single_benchmark(b6)
b6.close()

In [ ]:
# ── Бэкенд 7: TensorRT INT8 ──────────────────────────────────────────────────
if os.path.exists(TRT_INT8_PATH):
    b7 = TRTInt8Backend(TRT_INT8_PATH, batch_size=MAX_BATCH_SIZE)
    ALL_RESULTS["trt_int8"] = run_single_benchmark(b7)
    b7.close()
else:
    print(f"⚠  TRT INT8 движок не найден: {TRT_INT8_PATH}  — пропускаем")
    print("   Запустите ячейку 'TRT INT8 Build' в convert_to_onnx_trt.ipynb")

In [ ]:
# ── Бэкенд 3: ONNX CUDA ──────────────────────────────────────────────────
b3 = ONNXCUDABackend(ONNX_PATH, batch_size=MAX_BATCH_SIZE)
ALL_RESULTS["onnx_cuda"] = run_single_benchmark(b3)
b3.close()

In [ ]:
# ── Бэкенд 4: TensorRT FP16 ──────────────────────────────────────────────
b4 = TRTBackend(TRT_PATH, batch_size=MAX_BATCH_SIZE)
ALL_RESULTS["trt"] = run_single_benchmark(b4)
b4.close()

## Итоговая таблица

In [ ]:
def print_results_table(results: dict):
    rows = list(results.values())
    if not rows:
        print("Нет результатов")
        return

    # Базовая модель для расчёта speedup (первая)
    base_time = rows[0]["infer_s"]

    W = 100
    print("\n" + "═" * W)
    print(f" СРАВНЕНИЕ БЭКЕНДОВ CLIP-ReID (MSMT17 · {len(query_records)} query · {len(gallery_records)} gallery)")
    print("═" * W)

    # ── Качество ──────────────────────────────────────────────────────────
    print(f"\n{'Метрика качества':─^{W}}")
    hdr = f"  {'Модель':<18}  {'mAP':>7}  {'Rank-1':>7}  {'Rank-5':>7}  {'Rank-10':>7}"
    print(hdr)
    print("  " + "─" * (W-4))
    for r in rows:
        print(f"  {r['name']:<18}  {r['mAP']*100:>6.2f}%  "
              f"{r['rank1']*100:>6.2f}%  "
              f"{r['rank5']*100:>6.2f}%  "
              f"{r['rank10']*100:>6.2f}%")
    print(f"  {'paper (ViT-B/16)':<18}  {'69.90%':>7}  {'86.70%':>7}  {'93.70%':>7}  {'95.50%':>7}")

    # ── Precision / Recall / F2 ────────────────────────────────────────────
    for k in (1, 5, 10):
        print(f"\n{'Precision · Recall · F2  @K=' + str(k):─^{W}}")
        hdr2 = f"  {'Модель':<18}  {'P@'+str(k):>8}  {'R@'+str(k):>8}  {'F2@'+str(k):>8}"
        print(hdr2)
        print("  " + "─" * (W-4))
        for r in rows:
            p = r["prf"][k]
            print(f"  {r['name']:<18}  {p['precision']*100:>7.2f}%  "
                  f"{p['recall']*100:>7.2f}%  "
                  f"{p['f2']*100:>7.2f}%")

    # ── Скорость ──────────────────────────────────────────────────────────
    print(f"\n{'Скорость инференса':─^{W}}")
    hdr3 = f"  {'Модель':<18}  {'batch':>6}  {'ms/img':>8}  {'img/s':>8}  {'total s':>8}  {'speedup':>8}"
    print(hdr3)
    print("  " + "─" * (W-4))
    for r in rows:
        speedup = base_time / r["infer_s"]
        print(f"  {r['name']:<18}  {r['batch_size']:>6}  "
              f"{r['ms_per_img']:>8.3f}  "
              f"{r['throughput']:>8.0f}  "
              f"{r['infer_s']:>8.2f}  "
              f"{speedup:>7.2f}x")

    print("\n" + "═" * W)


print_results_table(ALL_RESULTS)

In [ ]:
# Сохранение результатов в CSV
import csv, datetime

ts  = datetime.datetime.now().strftime("%Y%m%d_%H%M")
csv_path = f"benchmark_results_{ts}.csv"

rows = []
for r in ALL_RESULTS.values():
    row = {
        "model":      r["name"],
        "batch":      r["batch_size"],
        "mAP":        round(r["mAP"] * 100, 3),
        "Rank1":      round(r["rank1"] * 100, 3),
        "Rank5":      round(r["rank5"] * 100, 3),
        "Rank10":     round(r["rank10"] * 100, 3),
    }
    for k in (1, 5, 10):
        p = r["prf"][k]
        row[f"P@{k}"]   = round(p["precision"] * 100, 3)
        row[f"R@{k}"]   = round(p["recall"]    * 100, 3)
        row[f"F2@{k}"]  = round(p["f2"]        * 100, 3)
    row["ms_per_img"]  = round(r["ms_per_img"],   4)
    row["throughput"]  = round(r["throughput"],   1)
    row["infer_s"]     = round(r["infer_s"],      2)
    rows.append(row)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"Результаты сохранены: {csv_path}")

In [ ]:
def print_distance_comparison(dist_results: dict):
    """
    Две таблицы:
    1. mAP × (бэкенд × метрика расстояния)
    2. Время вычисления матрицы расстояний
    """
    backends = list(dist_results.keys())
    methods  = DIST_METHODS_TO_RUN
    W = 90

    # ── Таблица 1: mAP ────────────────────────────────────────────────────
    for metric_key, metric_label in [
        ("mAP",   "mAP"),
        ("rank1", "Rank-1"),
    ]:
        print(f"\n{'═'*W}")
        print(f"  {metric_label}  ×  (бэкенд  ×  метрика расстояния)")
        print(f"{'═'*W}")
        col_w = 13
        header = f"  {'Бэкенд':<18}" + "".join(f"  {m:>{col_w}}" for m in methods)
        print(header)
        print("  " + "─" * (W - 2))
        for bk in backends:
            bname = dist_results[bk][methods[0]]   # чтобы достать имя
            row = f"  {ALL_RESULTS[bk]['name']:<18}"
            for m in methods:
                if m in dist_results[bk]:
                    val = dist_results[bk][m][metric_key] * 100
                    row += f"  {val:>{col_w}.2f}%"
                else:
                    row += f"  {'—':>{col_w}}"
            print(row)
        print(f"  {'paper':.<18}" + f"  {'69.90%':>{col_w}}" * len(methods))

    # ── Таблица 2: F2@1 ───────────────────────────────────────────────────
    print(f"\n{'═'*W}")
    print(f"  F2@1  ×  (бэкенд  ×  метрика расстояния)")
    print(f"{'═'*W}")
    print(header)
    print("  " + "─" * (W - 2))
    for bk in backends:
        row = f"  {ALL_RESULTS[bk]['name']:<18}"
        for m in methods:
            if m in dist_results[bk]:
                val = dist_results[bk][m]["prf"][1]["f2"] * 100
                row += f"  {val:>{col_w}.2f}%"
            else:
                row += f"  {'—':>{col_w}}"
        print(row)

    # ── Таблица 3: время вычисления матрицы ──────────────────────────────
    print(f"\n{'═'*W}")
    print(f"  Время вычисления distmat [сек]")
    print(f"{'═'*W}")
    print(header)
    print("  " + "─" * (W - 2))
    for bk in backends:
        row = f"  {ALL_RESULTS[bk]['name']:<18}"
        for m in methods:
            if m in dist_results[bk]:
                val = dist_results[bk][m]["dist_s"]
                row += f"  {val:>{col_w}.2f}s"
            else:
                row += f"  {'—':>{col_w}}"
        print(row)

    print(f"\n{'═'*W}")
    print("  Пояснения:")
    print("    euclidean = L2, GPU matmul, O(Q·G·D)")
    print("    cosine    = 1 - sim, GPU matmul, идентично L2 при нормализации")
    print("    manhattan = L1, GPU чанки по 12 запросов, медленнее")
    print("    reranking = k-reciprocal (Zhong 2017), CPU, +3-5% mAP")
    print(f"{'═'*W}\n")


print_distance_comparison(DIST_RESULTS)

In [ ]:
# Какие методы прогонять (reranking медленный — можно убрать из списка)
DIST_METHODS_TO_RUN = ["euclidean", "cosine", "manhattan", "reranking"]

# Накапливаем результаты: {backend_name: {dist_method: metrics_dict}}
DIST_RESULTS = {}

for backend_key, res in ALL_RESULTS.items():
    q_embs = res["q_embs"]
    g_embs = res["g_embs"]
    backend_name = res["name"]

    print(f"\n{'═'*60}")
    print(f"  {backend_name}")
    print(f"{'═'*60}")

    DIST_RESULTS[backend_key] = {}

    for method in DIST_METHODS_TO_RUN:
        print(f"\n  [{method}]")
        t0      = time.time()
        distmat, dist_s = compute_distmat(q_embs, g_embs, method)
        cmc, mAP = eval_func(distmat, q_pids, g_pids, q_camids, g_camids, max_rank=20)
        prf      = precision_recall_f2_at_k(
                       distmat, q_pids, g_pids, q_camids, g_camids, k_list=(1, 5, 10))
        total_s  = time.time() - t0

        DIST_RESULTS[backend_key][method] = {
            "mAP":    mAP,
            "rank1":  cmc[0],
            "rank5":  cmc[4],
            "rank10": cmc[9],
            "prf":    prf,
            "dist_s": dist_s,
            "total_s": total_s,
        }

        print(f"    mAP={mAP*100:.2f}%  R1={cmc[0]*100:.2f}%  "
              f"P@1={prf[1]['precision']*100:.2f}%  "
              f"F2@1={prf[1]['f2']*100:.2f}%  "
              f"({dist_s:.1f}s dist / {total_s:.1f}s total)")

        del distmat
        torch.cuda.empty_cache()

print("\nГотово ✓")

## Сравнение метрик расстояния

Для каждого бэкенда прогоняем все 4 метрики расстояния на уже извлечённых эмбеддингах.

| Метрика | Формула | GPU | Особенность |
|---|---|---|---|
| Euclidean (L2) | `2 - 2·(q·g)` | matmul | базовая, быстрая |
| Cosine | `1 - q·g` | matmul | = L2/2 при нормализации |
| Manhattan (L1) | `Σ\|qᵢ - gᵢ\|` | чанки по 12 | устойчива к выбросам |
| Re-ranking | k-reciprocal (Zhong 2017) | CPU | +3–5% mAP, медленно |

> **Примечание:** Euclidean и Cosine дают **идентичное ранжирование** при L2-нормализованных векторах (`dist_euc = 2 · dist_cos`). Разница в mAP между ними появляется только если эмбеддинги не нормализованы.